In [ ]:
import pandas as pd

hydrazide_df = pd.read_csv("../data/hydrazide_library.csv")
len(hydrazide_df)

In [ ]:
from rdkit import Chem

# Convert SMILES to RDKit molecules
hydrazide_df["mol"] = hydrazide_df["SMILES"].apply(Chem.MolFromSmiles)

# Count invalid molecules
invalid_count = hydrazide_df["mol"].isna().sum()
print(f"Invalid SMILES: {invalid_count}")

# Show invalid molecules (optional)
invalid_smiles = hydrazide_df[hydrazide_df["mol"].isna()]
display(invalid_smiles[["Name", "SMILES"]])

# Remove invalid molecules
hydrazide_df = hydrazide_df[
    hydrazide_df["mol"].notna()
].reset_index(drop=True)

len(hydrazide_df)

In [ ]:
hydrazide_pattern = Chem.MolFromSmarts("[CX3](=O)[NH][NH2]")

true_hydrazide_df = hydrazide_df[
    hydrazide_df["mol"].apply(
        lambda mol: mol.HasSubstructMatch(hydrazide_pattern)
    )
].reset_index(drop=True)

len(true_hydrazide_df)

In [ ]:
neutral_hydrazide_df = true_hydrazide_df[
    true_hydrazide_df["mol"].apply(
        lambda mol: Chem.GetFormalCharge(mol) == 0
    )
].reset_index(drop=True)

len(neutral_hydrazide_df)

In [ ]:
from rdkit.Chem import rdMolDescriptors

neutral_hydrazide_df["Aromatic_Ring_Count"] = hydrazide_df["mol"].apply(
    rdMolDescriptors.CalcNumAromaticRings
)
neutral_hydrazide_df["Aromatic_Ring_Count"].head()

In [ ]:
filtered_hydrazide_df = neutral_hydrazide_df[
    (neutral_hydrazide_df["Molecular_Weight"] < 500) &
    (neutral_hydrazide_df["XLogP"] < 5) &
    (neutral_hydrazide_df["Polar_Area"] < 140) &
    (neutral_hydrazide_df["Rotatable_Bond_Count"] < 10) &
    (neutral_hydrazide_df["Aromatic_Ring_Count"] >= 1)
].reset_index(drop=True)

len(filtered_hydrazide_df)

In [ ]:
functional_groups = {
    # Nitrogen heterocycles
    "Pyridine": Chem.MolFromSmarts("n1ccccc1"),
    "Pyrrole": Chem.MolFromSmarts("[nH]1cccc1"),
    "Imidazole": Chem.MolFromSmarts("c1ncc[nH]1"),
    "Pyrazole": Chem.MolFromSmarts("c1n[nH]cc1"),
    "1,2,3-Triazole": Chem.MolFromSmarts("n1nncc1"),
    "1,2,4-Triazole": Chem.MolFromSmarts("n1ncnc1"),
    "1,2,3,4-Tetrazole": Chem.MolFromSmarts("c1nnnn1"),
    "1-H-Tetrazole": Chem.MolFromSmarts("[nH]1nnnc1"),
    "Quinoline": Chem.MolFromSmarts("c1ccc2ncccc2c1"),
    "Isoquinoline": Chem.MolFromSmarts("c1ccc2ccnc2c1"),

    # Oxygen/Sulfur heterocycles
    "Thiophene": Chem.MolFromSmarts("c1ccsc1"),
    "Furan": Chem.MolFromSmarts("c1ccoc1"),
    "Benzofuran": Chem.MolFromSmarts("c1ccc2occc2c1"),

    # Functional groups
    "Phenol": Chem.MolFromSmarts("c[OH]"),
    "Methoxy": Chem.MolFromSmarts("[OX2][CH3]"),
    "Nitro": Chem.MolFromSmarts("[N+](=O)[O-]"),
    "Cyano": Chem.MolFromSmarts("C#N"),

    # Halogens
    "Fluoro": Chem.MolFromSmarts("[F]"),
    "Chloro": Chem.MolFromSmarts("[Cl]"),
    "Bromo": Chem.MolFromSmarts("[Br]"),
    "Iodo": Chem.MolFromSmarts("[I]"),

    # Carbonyl derivatives
    "Carboxylic_Acid": Chem.MolFromSmarts("C(=O)[OH]"),
    "Ester": Chem.MolFromSmarts("C(=O)O[#6]"),
    "Aldehyde": Chem.MolFromSmarts("[CX3H](=O)"),
    "Ketone": Chem.MolFromSmarts("[#6][CX3](=O)[#6]"),

    # Sulfur groups
    "Sulfone": Chem.MolFromSmarts("S(=O)(=O)"),
    "Sulfonamide": Chem.MolFromSmarts("S(=O)(=O)N"),

    # Medicinal chemistry
    "Trifluoromethyl": Chem.MolFromSmarts("C(F)(F)F")
}

for name, pattern in functional_groups.items():
    filtered_hydrazide_df[name] = filtered_hydrazide_df["mol"].apply(
        lambda mol: mol.HasSubstructMatch(pattern)
    )

fg_counts = filtered_hydrazide_df[
    functional_groups.keys()
].sum().sort_values(ascending=False)

display(fg_counts)

In [ ]:
import matplotlib.pyplot as plt

fg_counts.plot.bar(figsize=(12,5))

plt.ylabel("Number of molecules")
plt.title("Functional Group Distribution in Curated Hydrazide Library")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
hydrazide = Chem.MolFromSmarts("[CX3](=O)[NH][NH2]")
filtered_hydrazide_df["Hydrazide_Count"] = filtered_hydrazide_df["mol"].apply(
    lambda m: len(m.GetSubstructMatches(hydrazide))
)

filtered_hydrazide_df["Hydrazide_Count"].value_counts().sort_index()

In [ ]:
from rdkit.Chem import Draw

mols_2 = filtered_hydrazide_df[
    filtered_hydrazide_df["Hydrazide_Count"] == 1
]["mol"].tail(20).tolist()

legends_2 = filtered_hydrazide_df[
    filtered_hydrazide_df["Hydrazide_Count"] == 1
]["Name"].astype(str).tail(20).tolist()

Draw.MolsToGridImage(
    mols_2,
    legends=legends_2,
    molsPerRow=3,
    subImgSize=(300,300)
)

In [ ]:
hydrazide = Chem.MolFromSmarts("[CX3](=O)[NH][NH2]")

filtered_hydrazide_df["Is_Hydrazide"] = filtered_hydrazide_df["mol"].apply(
    lambda m: m is not None and m.HasSubstructMatch(hydrazide)
)

filtered_hydrazide_df["Is_Hydrazide"].value_counts()

In [ ]:
def has_aromatic_ring(mol):
    return any(atom.GetIsAromatic() for atom in mol.GetAtoms())

filtered_hydrazide_df["Aromatic"] = filtered_hydrazide_df["mol"].apply(has_aromatic_ring)

filtered_hydrazide_df["Aromatic"].value_counts()

In [ ]:
def has_aromatic_ring(mol):
    ring_info = mol.GetRingInfo()
    for ring in ring_info.AtomRings():
        if all(mol.GetAtomWithIdx(i).GetIsAromatic() for i in ring):
            return True
    return False

filtered_hydrazide_df = filtered_hydrazide_df[
    filtered_hydrazide_df["mol"].apply(has_aromatic_ring)
].copy()

len(filtered_hydrazide_df)

In [ ]:
filtered_hydrazide_df.to_csv(
    "../data/clean_hydrazides.csv",
    index=False
)